# Crete Acceptability Experiment
**Πανεπιστήμιο Κρήτης — Chatzikyriakidis 2026**

Τρέξτε τα κελιά με τη σειρά (Shift+Enter). Κάθε κελί εξηγεί τι κάνει.

## 1. Εγκατάσταση (μόνο την πρώτη φορά)

In [ ]:
!pip install openai

## 2. API Key
Βάλτε το κλειδί που σας έδωσε ο Στέργιος. **Τρέξτε αυτό κάθε φορά** που ανοίγετε το notebook.

In [ ]:
import os
os.environ['AZURE_KEY'] = 'PASTE-YOUR-KEY-HERE'  # <-- αλλάξτε αυτό

## 3. Πηγαίνετε στον φάκελο του πειράματος
Αν το notebook είναι ήδη μέσα στον φάκελο `crete_experiment/`, παραλείψτε αυτό το κελί.

In [ ]:
# Αλλάξτε τη διαδρομή αν χρειαστεί
import os
if os.path.basename(os.getcwd()) != 'crete_experiment':
    # Προσπαθεί να βρει τον φάκελο
    if os.path.isdir('crete_experiment'):
        os.chdir('crete_experiment')
    else:
        print('⚠ Δεν βρέθηκε ο φάκελος crete_experiment/')
        print('  Τρέχον directory:', os.getcwd())
        print('  Βάλτε τη σωστή διαδρομή παρακάτω:')
        # os.chdir('/path/to/crete_experiment')  # <-- uncomment και βάλτε τη σωστή διαδρομή

print('Working directory:', os.getcwd())
print('Stimuli files:', os.listdir('stimuli') if os.path.isdir('stimuli') else '⚠ not found')

## 4. Δοκιμαστική εκτέλεση (Dry Run)
Ελέγχει ότι τα stimuli φορτώνονται σωστά, **χωρίς** να καλεί το API (δεν κοστίζει).

In [ ]:
%run run_experiment.py --dry-run --reps 2

### Αν βλέπετε σφάλματα:
- `JSONDecodeError` → Ανοίξτε το αρχείο .jsonl και διορθώστε τα εισαγωγικά/κόμματα
- `Λείπουν τα πεδία` → Ελέγξτε ότι κάθε γραμμή έχει: id, sentence, phenomenon, condition

Διορθώστε τα σφάλματα και ξανατρέξτε το κελί μέχρι να δουλέψει.

## 5. Κανονική εκτέλεση
Αφού δουλέψει το dry run, τρέξτε κανονικά. **Αυτό κοστίζει API credits** — μην τρέχετε χωρίς λόγο.

Ξεκινήστε δοκιμαστικά με **ένα αρχείο** και **λίγες επαναλήψεις**:

In [ ]:
# Δοκιμή: 1 αρχείο, 1 μοντέλο, 3 reps
%run run_experiment.py --file stimuli/cd.jsonl --models gpt-4o --reps 3

Αν δουλέψει, τρέξτε τα πάντα (θα πάρει 30-60 λεπτά):

In [ ]:
# Πλήρες πείραμα — ΜΟΝΟ αφού είστε σίγουροι
# Αποσχολιάστε (βγάλτε το #) και τρέξτε:

# %run run_experiment.py

## 6. Ανάλυση αποτελεσμάτων
Βρείτε το αρχείο αποτελεσμάτων και τρέξτε την ανάλυση:

In [ ]:
# Δείτε ποια αρχεία αποτελεσμάτων υπάρχουν
import glob
result_files = sorted(glob.glob('results/results_*.json'))
for f in result_files:
    print(f)

In [ ]:
# Τρέξτε ανάλυση στο τελευταίο αρχείο
if result_files:
    latest = result_files[-1]
    print(f'Analyzing: {latest}')
    %run analyze_results.py {latest}
else:
    print('Δεν βρέθηκαν αποτελέσματα. Τρέξτε πρώτα το πείραμα (Βήμα 5).')

## 7. Εξερεύνηση αποτελεσμάτων στο notebook
Φορτώστε τα αποτελέσματα σαν Python dictionary για να τα εξερευνήσετε:

In [ ]:
import json

# Φορτώστε το τελευταίο αρχείο αποτελεσμάτων
if result_files:
    with open(result_files[-1]) as f:
        data = json.load(f)
    
    print(f"Σύνολο: {len(data['results'])} αποτελέσματα")
    print(f"Μοντέλα: {set(r['model'] for r in data['results'])}")
    print(f"Φαινόμενα: {set(r['phenomenon'] for r in data['results'])}")
    print()
    
    # Δείξε τα πρώτα 10
    print(f"{'ID':<15} {'Condition':<25} {'Model':<15} {'Mean':>5} {'SD':>5}")
    print('-' * 70)
    for r in data['results'][:10]:
        mean = f"{r['mean']:.1f}" if r['mean'] else '—'
        sd = f"{r['sd']:.1f}" if r['sd'] else '—'
        print(f"{r['id']:<15} {r['condition']:<25} {r['model']:<15} {mean:>5} {sd:>5}")

## 8. Γράφημα (προαιρετικό)
Αν θέλετε να δείτε τα αποτελέσματα σε γράφημα:

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    !pip install matplotlib
    import matplotlib.pyplot as plt

if result_files:
    # Επιλέξτε φαινόμενο
    PHENOMENON = 'cd'  # αλλάξτε σε: clld, crossover, binding, plural_conjunction
    
    # Φιλτράρισμα
    filtered = [r for r in data['results'] 
                if r['phenomenon'] == PHENOMENON and r['mean'] is not None]
    
    if not filtered:
        print(f'Δεν βρέθηκαν αποτελέσματα για {PHENOMENON}')
    else:
        # Ομαδοποίηση κατά μοντέλο
        models = sorted(set(r['model'] for r in filtered))
        conditions = sorted(set(r['condition'] for r in filtered))
        
        fig, ax = plt.subplots(figsize=(12, 5))
        x = range(len(conditions))
        width = 0.8 / len(models)
        
        for i, model in enumerate(models):
            means = []
            for cond in conditions:
                match = [r for r in filtered if r['model'] == model and r['condition'] == cond]
                means.append(match[0]['mean'] if match else 0)
            offset = (i - len(models)/2 + 0.5) * width
            ax.bar([xi + offset for xi in x], means, width, label=model[:12])
        
        ax.set_xticks(x)
        ax.set_xticklabels(conditions, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Mean Rating (1-7)')
        ax.set_title(f'{PHENOMENON.upper()} — Mean Ratings by Model')
        ax.legend(fontsize=8)
        ax.set_ylim(0, 7.5)
        plt.tight_layout()
        plt.show()